In [1]:
!pip install ipeadatapy
import pandas as pd
import numpy as np
import re
import unicodedata
import ipeadatapy
import requests


municipios_rmsp = [
    2513851,
    3503901,
    3509007,
    3509205,
    3510609,
    3513009,
    3513801,
    3515103,
    3515707,
    3516309,
    3516408,
    3518305,
    3518800,
    3522505,
    3523107,
    3525003,
    3526209,
    3528502,
    3529401,
    3530607,
    3534401,
    3539103,
    3539806,
    3543303,
    3544103,
    3545001,
    3546801,
    3547304,
    3547809,
    3548708,
    3548807,
    3549953,
    3550308,
    3552502,
    3552809,
    3556453,
    5219357,
]

# CARREGAR DADOS
def to_snake_case(column_name):
    """
    Converte um nome de coluna para snake_case, removendo acentos e ç.

    Args:
        column_name (str): Nome da coluna original

    Returns:
        str: Nome da coluna em snake_case sem acentos
    """
    # Remove parênteses e seu conteúdo
    column_name = re.sub(r"\s*\([^)]*\)", "", column_name)

    # Remove acentos e normaliza caracteres
    column_name = unicodedata.normalize("NFD", column_name)
    column_name = "".join(c for c in column_name if not unicodedata.combining(c))

    # Converte para minúsculas
    column_name = column_name.lower()

    # Substitui ç por c
    column_name = column_name.replace("ç", "c")

    # Substitui espaços e caracteres especiais por underscore
    column_name = re.sub(r"[^\w\s]", " ", column_name)
    column_name = re.sub(r"\s+", "_", column_name)

    # Remove underscores duplicados e no início/fim
    column_name = re.sub(r"_+", "_", column_name)
    column_name = column_name.strip("_")

    return column_name


def transform_columns_to_snake_case(df):
    """
    Transforma todas as colunas de um DataFrame para snake_case.

    Args:
        df (pandas.DataFrame): DataFrame com as colunas a serem transformadas

    Returns:
        pandas.DataFrame: DataFrame com colunas em snake_case
    """
    # Cria um mapeamento das colunas originais para snake_case
    column_mapping = {col: to_snake_case(col) for col in df.columns}

    # Renomeia as colunas
    df_renamed = df.rename(columns=column_mapping)

    return df_renamed


def extrair_pop_censo_api():
    url_pop_censo = (
        "https://apisidra.ibge.gov.br/values"
        "/t/9514"
        "/p/2022"
        "/v/93"
        "/n6/3534401"
        "/c2/4,5"
        "/c287/93070,93084,93085,93086,93087,93088,93089,93090,93091,93092,93093,93094,93095,93096,93097,93098,49108,49109,60040,60041,6653"
    )
    pop_osasco_censo22 = pd.read_json(url_pop_censo)

    # tratamentos
    pop_osasco_censo22.columns = pop_osasco_censo22.iloc[0]
    pop_osasco_censo22 = pop_osasco_censo22.drop(pop_osasco_censo22.index[0])

    pop_osasco_censo22["Valor"] = pop_osasco_censo22["Valor"].astype(int)

    colunas_a_remover = pop_osasco_censo22.filter(like="(Código)").columns
    pop_osasco_censo22 = pop_osasco_censo22.drop(columns=colunas_a_remover)

    pop_osasco_censo22 = transform_columns_to_snake_case(pop_osasco_censo22)

    return pop_osasco_censo22


def extrair_pop_urbana_rural_ipedatapy():
    def extrair_serie_pop_urbana_rural(serie):
        pop = ipeadatapy.timeseries(serie).reset_index(drop=True)
        pop.columns = (
            pop.columns.str.lower()
            .str.replace("(", "")
            .str.replace(")", "")
            .str.replace(" ", "_")
        )
        pop = pop[["tercodigo", "year", "value_habitante"]]
        pop["categoria"] = "pop_urbana" if serie == "POPUR" else "pop_rural"
        return pop

    def processar_tratar():
        pop_urbana = extrair_serie_pop_urbana_rural("POPUR")
        pop_rural = extrair_serie_pop_urbana_rural("POPRU")

        munic_selecionados = {
            "3548708": "São Bernardo do Campo",
            "3534401": "Osasco",
            "3552205": "Sorocaba",
            "3543402": "Ribeirão Preto",
            "3547809": "Santo André",
            "3549904": "São José dos Campos",
        }
        pop_urbana["municipio"] = pop_urbana["tercodigo"].map(munic_selecionados)
        pop_rural["municipio"] = pop_rural["tercodigo"].map(munic_selecionados)

        pop_urbana = pop_urbana[pop_urbana["tercodigo"].isin(munic_selecionados.keys())]
        pop_rural = pop_rural[pop_rural["tercodigo"].isin(munic_selecionados.keys())]

        pop = pd.concat([pop_urbana, pop_rural], axis=0)

        pop = pop.pivot(
            index=["tercodigo", "year", "municipio"],
            columns="categoria",
            values="value_habitante",
        ).reset_index()

        pop = pop.query("year > 1960")
        return pop

    def calcular_proporcoes(df):
        df[["pop_rural", "pop_urbana"]] = df[["pop_rural", "pop_urbana"]].fillna(0)
        df["pop_total"] = df["pop_urbana"] + df["pop_rural"]
        df["prop_urbana"] = df["pop_urbana"] / df["pop_total"]
        df["prop_rural"] = df["pop_rural"] / df["pop_total"]
        return df

    pop = processar_tratar()
    pop = calcular_proporcoes(pop)
    return pop


def extrair_pop_envelhecimento_api():
    url_pop_envelhecimento = (
        "https://apisidra.ibge.gov.br/values"
        "/t/9756"
        "/p/2010,2022"
        "/v/9175,10613,8845"
        "/n6/3534401,3548708,3552205,3543402,3547809,3549904"
        "/c86/95251,2776,2777,2778,2779,2780"
    )
    pop_osasco_envelhecimento = pd.read_json(url_pop_envelhecimento)

    # tratamentos
    pop_osasco_envelhecimento.columns = pop_osasco_envelhecimento.iloc[0]
    pop_osasco_envelhecimento = pop_osasco_envelhecimento.drop(
        pop_osasco_envelhecimento.index[0]
    )

    colunas_a_remover = pop_osasco_envelhecimento.filter(like="(Código)").columns
    pop_osasco_envelhecimento = pop_osasco_envelhecimento.drop(
        columns=colunas_a_remover
    )

    pop_osasco_envelhecimento = transform_columns_to_snake_case(
        pop_osasco_envelhecimento
    )

    envelhecimento = pop_osasco_envelhecimento.query(
        "variavel == 'Índice de envelhecimento (Idosos: 60 anos ou mais de idade)'"
    ).drop(columns=["nivel_territorial", "unidade_de_medida"])

    envelhecimento["valor"] = envelhecimento["valor"].astype(float)

    envelhecimento["municipio"] = envelhecimento["municipio"].replace(
        {
            "Osasco (SP)": "Osasco",
            "São Bernardo do Campo (SP)": "São Bernardo do Campo",
            "Sorocaba (SP)": "Sorocaba",
            "Ribeirão Preto (SP)": "Ribeirão Preto",
            "Santo André (SP)": "Santo André",
            "São José dos Campos (SP)": "São José dos Campos",
        }
    )

    return envelhecimento


def extrair_dependencia_demografica_api():

    def extrair_pop_censo22_api():
        url_pop_censo = (
            "https://apisidra.ibge.gov.br/values"
            "/t/9514"
            "/p/2022"
            "/v/93"
            "/n6/3534401,3548708,3552205,3543402,3547809,3549904"
            "/c2/4,5"
            "/c287/93070,93084,93085,93086,93087,93088,93089,93090,93091,93092,93093,93094,93095,93096,93097,93098,49108,49109,60040,60041,6653"
        )
        pop22 = pd.read_json(url_pop_censo)

        # tratamentos
        pop22.columns = pop22.iloc[0]
        pop22 = pop22.drop(pop22.index[0])

        pop22["Valor"] = pop22["Valor"].astype(int)

        colunas_a_remover = pop22.filter(like="(Código)").columns.tolist()
        pop22 = pop22.drop(columns=colunas_a_remover)

        pop22 = transform_columns_to_snake_case(pop22)
        pop22 = pop22.drop(columns=["forma_de_declaracao_da_idade"])

        map_populacao_inativa_ativa22 = {
            "0 a 4 anos": "inativa",
            "5 a 9 anos": "inativa",
            "10 a 14 anos": "inativa",
            "15 a 19 anos": "ativa",
            "20 a 24 anos": "ativa",
            "25 a 29 anos": "ativa",
            "30 a 34 anos": "ativa",
            "35 a 39 anos": "ativa",
            "40 a 44 anos": "ativa",
            "45 a 49 anos": "ativa",
            "50 a 54 anos": "ativa",
            "55 a 59 anos": "ativa",
            "60 a 64 anos": "ativa",
            "65 a 69 anos": "inativa",
            "70 a 74 anos": "inativa",
            "75 a 79 anos": "inativa",
            "80 a 84 anos": "inativa",
            "85 a 89 anos": "inativa",
            "90 a 94 anos": "inativa",
            "95 a 99 anos": "inativa",
            "100 anos ou mais": "inativa",
        }
        pop22["populacao_inativa_ativa"] = pop22["idade"].map(
            map_populacao_inativa_ativa22
        )

        pop22 = pop22.groupby(
            ["ano", "municipio", "sexo", "populacao_inativa_ativa"], as_index=False
        ).agg({"valor": "sum"})

        return pop22

    def extrair_pop_censo0010_api():
        url_pop_censo0010 = (
            "https://apisidra.ibge.gov.br/values"
            "/t/1552"
            "/p/2000,2010"
            "/v/93"
            "/n6/3534401,3548708,3552205,3543402,3547809,3549904"
            "/c2/92956,92957"
            "/c287/93070,93084,93085,93086,93087,93088,93089,93090,93091,93092,93093,93094,93095,93096,93097,93098,93099,93100,6653"
        )
        pop0010 = pd.read_json(url_pop_censo0010)

        # tratamentos
        pop0010.columns = pop0010.iloc[0]
        pop0010 = pop0010.drop(pop0010.index[0])

        pop0010["Valor"] = pop0010["Valor"].astype(int)

        colunas_a_remover = pop0010.filter(like="(Código)").columns.tolist()
        pop0010 = pop0010.drop(columns=colunas_a_remover)

        pop0010 = transform_columns_to_snake_case(pop0010)

        pop0010 = pop0010.drop(
            columns=["situacao_do_domicilio", "forma_de_declaracao_da_idade"]
        )

        pop0010["sexo"] = np.where(pop0010["sexo"] == "Homem", "Homens", "Mulheres")

        map_populacao_inativa_ativa0010 = {
            "0 a 4 anos": "inativa",
            "5 a 9 anos": "inativa",
            "10 a 14 anos": "inativa",
            "15 a 19 anos": "ativa",
            "20 a 24 anos": "ativa",
            "25 a 29 anos": "ativa",
            "30 a 34 anos": "ativa",
            "35 a 39 anos": "ativa",
            "40 a 44 anos": "ativa",
            "45 a 49 anos": "ativa",
            "50 a 54 anos": "ativa",
            "55 a 59 anos": "ativa",
            "60 a 64 anos": "ativa",
            "65 a 69 anos": "inativa",
            "70 a 74 anos": "inativa",
            "75 a 79 anos": "inativa",
            "80 a 89 anos": "inativa",
            "90 a 99 anos": "inativa",
            "100 anos ou mais": "inativa",
        }

        pop0010["populacao_inativa_ativa"] = pop0010["idade"].map(
            map_populacao_inativa_ativa0010
        )

        pop0010 = pop0010.groupby(
            ["ano", "municipio", "sexo", "populacao_inativa_ativa"], as_index=False
        ).agg({"valor": "sum"})

        return pop0010

    pop_censo0010 = extrair_pop_censo0010_api()
    pop_censo22 = extrair_pop_censo22_api()
    pop_ativa_inativa = pd.concat([pop_censo0010, pop_censo22], axis=0)

    pop_ativa_inativa_prop = pop_ativa_inativa.groupby(
        ["ano", "municipio", "populacao_inativa_ativa"], as_index=False
    ).agg({"valor": "sum"})

    pop_ativa_inativa_prop = pop_ativa_inativa_prop.pivot_table(
        index=["ano", "municipio"], columns="populacao_inativa_ativa", values="valor"
    ).reset_index()

    pop_ativa_inativa_prop["prop_inativa"] = (
        pop_ativa_inativa_prop["inativa"] / pop_ativa_inativa_prop["ativa"]
    )

    pop_ativa_inativa_prop["municipio"] = pop_ativa_inativa_prop["municipio"].replace(
        {
            "Osasco (SP)": "Osasco",
            "São Bernardo do Campo (SP)": "São Bernardo do Campo",
            "Sorocaba (SP)": "Sorocaba",
            "Ribeirão Preto (SP)": "Ribeirão Preto",
            "Santo André (SP)": "Santo André",
            "São José dos Campos (SP)": "São José dos Campos",
        }
    )

    return pop_ativa_inativa_prop


def extrair_pop_genero_api():


    def extrair_pop_censo22_api():
        url_pop_censo = (
            "https://apisidra.ibge.gov.br/values"
            "/t/9514"
            "/p/2022"
            "/v/93"
            "/n6/3534401,3548708,3552205,3543402,3547809,3549904"
            "/c2/4,5"
            "/c287/93070,93084,93085,93086,93087,93088,93089,93090,93091,93092,93093,93094,93095,93096,93097,93098,49108,49109,60040,60041,6653"
        )
        pop22 = pd.read_json(url_pop_censo)

        # tratamentos
        pop22.columns = pop22.iloc[0]
        pop22 = pop22.drop(pop22.index[0])

        pop22["Valor"] = pop22["Valor"].astype(int)

        colunas_a_remover = pop22.filter(like="(Código)").columns.tolist()
        pop22 = pop22.drop(columns=colunas_a_remover)

        pop22 = transform_columns_to_snake_case(pop22)
        pop22 = pop22.drop(columns=["forma_de_declaracao_da_idade"])

        map_populacao_inativa_ativa22 = {
            "0 a 4 anos": "inativa",
            "5 a 9 anos": "inativa",
            "10 a 14 anos": "inativa",
            "15 a 19 anos": "ativa",
            "20 a 24 anos": "ativa",
            "25 a 29 anos": "ativa",
            "30 a 34 anos": "ativa",
            "35 a 39 anos": "ativa",
            "40 a 44 anos": "ativa",
            "45 a 49 anos": "ativa",
            "50 a 54 anos": "ativa",
            "55 a 59 anos": "ativa",
            "60 a 64 anos": "ativa",
            "65 a 69 anos": "inativa",
            "70 a 74 anos": "inativa",
            "75 a 79 anos": "inativa",
            "80 a 84 anos": "inativa",
            "85 a 89 anos": "inativa",
            "90 a 94 anos": "inativa",
            "95 a 99 anos": "inativa",
            "100 anos ou mais": "inativa",
        }
        pop22["populacao_inativa_ativa"] = pop22["idade"].map(
            map_populacao_inativa_ativa22
        )

        pop22 = pop22.groupby(
            ["ano", "municipio", "sexo", "populacao_inativa_ativa"], as_index=False
        ).agg({"valor": "sum"})

        return pop22

    def extrair_pop_censo0010_api():
        url_pop_censo0010 = (
            "https://apisidra.ibge.gov.br/values"
            "/t/1552"
            "/p/2000,2010"
            "/v/93"
            "/n6/3534401,3548708,3552205,3543402,3547809,3549904"
            "/c2/92956,92957"
            "/c287/93070,93084,93085,93086,93087,93088,93089,93090,93091,93092,93093,93094,93095,93096,93097,93098,93099,93100,6653"
        )
        pop0010 = pd.read_json(url_pop_censo0010)

        # tratamentos
        pop0010.columns = pop0010.iloc[0]
        pop0010 = pop0010.drop(pop0010.index[0])

        pop0010["Valor"] = pop0010["Valor"].astype(int)

        colunas_a_remover = pop0010.filter(like="(Código)").columns.tolist()
        pop0010 = pop0010.drop(columns=colunas_a_remover)

        pop0010 = transform_columns_to_snake_case(pop0010)

        pop0010 = pop0010.drop(
            columns=["situacao_do_domicilio", "forma_de_declaracao_da_idade"]
        )

        pop0010["sexo"] = np.where(pop0010["sexo"] == "Homem", "Homens", "Mulheres")

        map_populacao_inativa_ativa0010 = {
            "0 a 4 anos": "inativa",
            "5 a 9 anos": "inativa",
            "10 a 14 anos": "inativa",
            "15 a 19 anos": "ativa",
            "20 a 24 anos": "ativa",
            "25 a 29 anos": "ativa",
            "30 a 34 anos": "ativa",
            "35 a 39 anos": "ativa",
            "40 a 44 anos": "ativa",
            "45 a 49 anos": "ativa",
            "50 a 54 anos": "ativa",
            "55 a 59 anos": "ativa",
            "60 a 64 anos": "ativa",
            "65 a 69 anos": "inativa",
            "70 a 74 anos": "inativa",
            "75 a 79 anos": "inativa",
            "80 a 89 anos": "inativa",
            "90 a 99 anos": "inativa",
            "100 anos ou mais": "inativa",
        }

        pop0010["populacao_inativa_ativa"] = pop0010["idade"].map(
            map_populacao_inativa_ativa0010
        )

        pop0010 = pop0010.groupby(
            ["ano", "municipio", "sexo", "populacao_inativa_ativa"], as_index=False
        ).agg({"valor": "sum"})

        return pop0010

    pop_censo0010 = extrair_pop_censo0010_api()
    pop_censo22 = extrair_pop_censo22_api()
    pop_ativa_inativa = pd.concat([pop_censo0010, pop_censo22], axis=0)

    pop_genero = pop_ativa_inativa.groupby(
        ["ano", "municipio", "sexo"], as_index=False
    ).agg({"valor": "sum"})
    pop_genero["total_pop"] = pop_genero.groupby(["ano", "municipio"])[
        "valor"
    ].transform("sum")
    pop_genero["prop_genero"] = pop_genero["valor"] / pop_genero["total_pop"]

    pop_genero["municipio"] = pop_genero["municipio"].replace(
        {
            "Osasco (SP)": "Osasco",
            "São Bernardo do Campo (SP)": "São Bernardo do Campo",
            "Sorocaba (SP)": "Sorocaba",
            "Ribeirão Preto (SP)": "Ribeirão Preto",
            "Santo André (SP)": "Santo André",
            "São José dos Campos (SP)": "São José dos Campos",
        }
    )

    return pop_genero


def extrair_6_17_frequenta_escola_api():
    url_pop_censo = (
        "https://apisidra.ibge.gov.br/values"
        "/t/10058"
        "/p/2022"
        "/v/13283"
        "/n6/3534401,3548708,3552205,3543402,3547809,3549904"
        "/c58/2489,2490,2491,2493,2494,2495,2496,24972498,2499,2500"
        "/c11798/95301,107454,7905,7906,7907,7908,7909,95307"
        "/c2/4,5"
        "/c86/2776,2777,2778,2779,2780"
    )
    df = pd.read_json(url_pop_censo)

    # tratamentos
    df.columns = df.iloc[0]
    df = df.drop(df.index[0])

    df["Valor"] = df["Valor"].str.replace("-", "0").str.replace("...", "0").astype(int)

    colunas_a_remover = df.filter(like="(Código)").columns.tolist()
    df = df.drop(columns=colunas_a_remover)

    df = transform_columns_to_snake_case(df)

    df = (
        df.groupby(
            ["municipio", "nivel_de_ensino_ou_curso_que_frequentavam"], as_index=False
        )
        .agg({"valor": "sum"})
        .sort_values("valor", ascending=True)
    )

    df["municipio"] = df["municipio"].replace(
        {
            "Osasco (SP)": "Osasco",
            "São Bernardo do Campo (SP)": "São Bernardo do Campo",
            "Sorocaba (SP)": "Sorocaba",
            "Ribeirão Preto (SP)": "Ribeirão Preto",
            "Santo André (SP)": "Santo André",
            "São José dos Campos (SP)": "São José dos Campos",
        }
    )

    return df


def extrair_moradores_domicilio_api():
    url_pop_censo = (
        "https://apisidra.ibge.gov.br/values"
        "/t/6893"
        "/p/2022"
        "/v/382"
        "/n6/3534401,3548708,3552205,3543402,3547809,3549904"
        "/c125/2932,6815,121264,3247,71975,71976,71977"  # tipo de domicilio
        "/c86/95251,2776,2777,2778,2779,2780"  # cor ou raça
    )
    df = pd.read_json(url_pop_censo)

    # tratamentos
    df.columns = df.iloc[0]
    df = df.drop(df.index[0])

    df["Valor"] = df["Valor"].str.replace("-", "0").str.replace("...", "0").astype(int)

    colunas_a_remover = df.filter(like="(Código)").columns.tolist()
    df = df.drop(columns=colunas_a_remover)

    df = transform_columns_to_snake_case(df)

    moradores_grp = (
        df.query("cor_ou_raca == 'Total' and tipo_de_domicilio != 'Total'")
        .sort_values("valor", ascending=False)
        .groupby(["municipio", "tipo_de_domicilio"], as_index=False)
        .agg({"valor": "sum"})
    )
    moradores_grp["valor_total"] = moradores_grp.groupby("municipio")[
        "valor"
    ].transform("sum")
    moradores_grp["prop_moradores"] = (
        moradores_grp["valor"] / moradores_grp["valor_total"]
    )

    moradores_grp = moradores_grp.drop(columns=["valor_total"])
    return moradores_grp


def extrair_renda_domicilio_api_2010():
    def extrair_3578_2010(municipio, timeout=60):
        base = "https://apisidra.ibge.gov.br/values"
        path = "/t/3578/p/2010/v/2035"
        classes = "/c386/12009,12010,9682,9683,9684,9685,9686,9687,9691,9692"
        url = f"{base}{path}{classes}/n6/{municipio}"
        resp = requests.get(url, timeout=timeout)
        resp.raise_for_status()
        data = resp.json()
        return pd.DataFrame(data)

    dfs = []
    for munic in municipios_rmsp:
        df = extrair_3578_2010(munic)
        dfs.append(df)

    df_completo = pd.concat(dfs, axis=0)

    # tratamentos
    df_completo.columns = df_completo.iloc[0]
    df_completo = df_completo.drop(df_completo.index[0])

    df_completo["Valor"] = (
        df_completo["Valor"].str.replace("-", "0").str.replace("...", "0").astype(float)
    )

    colunas_a_remover = df_completo.filter(like="(Código)").columns.tolist()
    df_completo = df_completo.drop(columns=colunas_a_remover)

    df_completo = transform_columns_to_snake_case(df_completo)

    renda2010_grp = (
        df_completo.query(
            "classes_de_rendimento_nominal_mensal_domiciliar_per_capita != 'Total' and variavel != 'Domicílios particulares permanentes'"
        )
        .groupby(
            ["municipio", "classes_de_rendimento_nominal_mensal_domiciliar_per_capita"],
            as_index=False,
        )
        .agg({"valor": "sum"})
    )

    return renda2010_grp


def extrair_renda_domicilio_api_2022():

    def extrair_10296_2022(municipio, timeout=60):
        base = "https://apisidra.ibge.gov.br/values"
        path = "/t/10296/p/2022/v/13604"
        classes = "/c386/9681,9682,9683,9684,9685,9686,9687,9688,9689,9690,9692"
        url = f"{base}{path}{classes}/n6/{municipio}"
        resp = requests.get(url, timeout=timeout)
        resp.raise_for_status()
        data = resp.json()
        return pd.DataFrame(data)

    dfs = []
    for munic in municipios_rmsp:
        df = extrair_10296_2022(munic)
        dfs.append(df)

    df_completo = pd.concat(dfs, axis=0)

    # tratamentos
    df_completo.columns = df_completo.iloc[0]
    df_completo = df_completo.drop(df_completo.index[0])

    df_completo["Valor"] = (
        df_completo["Valor"].str.replace("-", "0").str.replace("...", "0").astype(float)
    )

    colunas_a_remover = df_completo.filter(like="(Código)").columns.tolist()
    df_completo = df_completo.drop(columns=colunas_a_remover)

    df_completo = transform_columns_to_snake_case(df_completo)

    renda2022_grp = (
        df_completo.query(
            "classes_de_rendimento_nominal_mensal_domiciliar_per_capita != 'Total' and variavel != 'Domicílios particulares permanentes'"
        )
        .groupby(
            ["municipio", "classes_de_rendimento_nominal_mensal_domiciliar_per_capita"],
            as_index=False,
        )
        .agg({"valor": "sum"})
    )

    return renda2022_grp


def extrair_fecundidade_api():
    url_pop_censo = (
        "https://apisidra.ibge.gov.br/values"
        "/t/10076"
        "/p/2010,2022"
        "/v/13314"
        "/n6/3534401,3548708,3552205,3543402,3547809,3549904"
        "/c12291/58895,105103,105104,105105,105106,105107,105108"
        "/c12293/58898,105167,105168,105169,105170,105171,105172"
        "/c12232/58896,58897,104541,104542,104543,104544,104545,104546,104547,104548,104549,105109,105110,105111,105112,105113"
    )
    df = pd.read_json(url_pop_censo)

    # tratamentos
    df.columns = df.iloc[0]
    df = df.drop(df.index[0])

    df["Valor"] = (
        df["Valor"].str.replace("-", "0").str.replace("...", "0").astype(float)
    )

    colunas_a_remover = df.filter(like="(Código)").columns.tolist()
    df = df.drop(columns=colunas_a_remover)

    df = transform_columns_to_snake_case(df)

    fecundidade_grp = (
        df.query("numero_de_filhos_tidos_nascidos_vivos == 'Total' and cor_ou_raca_das_mulheres == 'Total' and grupos_de_idade_das_mulheres != 'Total'")
        .groupby(
            [
                "ano",
                "municipio",
                "grupos_de_idade_das_mulheres",
            ],
            as_index=False,
        )
        .agg({"valor": "sum"})
    )

    fecundidade_grp["municipio"] = fecundidade_grp["municipio"].replace(
        {
            "Osasco (SP)": "Osasco",
            "São Bernardo do Campo (SP)": "São Bernardo do Campo",
            "Sorocaba (SP)": "Sorocaba",
            "Ribeirão Preto (SP)": "Ribeirão Preto",
            "Santo André (SP)": "Santo André",
            "São José dos Campos (SP)": "São José dos Campos",
        }
    )

    return fecundidade_grp


def extrair_moradores_domicilio_cor_raca_api():
    url_pop_censo = (
        "https://apisidra.ibge.gov.br/values"
        "/t/6893"
        "/p/2022"
        "/v/382"
        "/n6/3534401,3548708,3552205,3543402,3547809,3549904"
        "/c125/2932,6815,121264,3247,71975,71976,71977" # tipo de domicilio
        "/c86/95251,2776,2777,2778,2779,2780" # cor ou raça
    )
    df = pd.read_json(url_pop_censo)

    # tratamentos
    df.columns = df.iloc[0]
    df = df.drop(df.index[0])

    df["Valor"] = (
        df["Valor"].str.replace("-", "0").str.replace("...", "0").astype(int)
    )

    colunas_a_remover = df.filter(like="(Código)").columns.tolist()
    df = df.drop(columns=colunas_a_remover)

    df = transform_columns_to_snake_case(df)

    df = (
        df.loc[(df['tipo_de_domicilio'] != 'Habitação indígena sem paredes ou maloca') & (df['cor_ou_raca'] != 'Total')]
    )

    df['pop_total'] = df.groupby(['municipio', 'tipo_de_domicilio'], as_index=False)['valor'].transform('sum')
    df['prop'] = df['valor'] / df['pop_total']

    df = df.sort_values(["tipo_de_domicilio", "prop"], ascending=False)


    df["municipio"] = df["municipio"].replace(
        {
            "Osasco (SP)": "Osasco",
            "São Bernardo do Campo (SP)": "São Bernardo do Campo",
            "Sorocaba (SP)": "Sorocaba",
            "Ribeirão Preto (SP)": "Ribeirão Preto",
            "Santo André (SP)": "Santo André",
            "São José dos Campos (SP)": "São José dos Campos",
        }
    )

    return df[['ano', 'municipio', 'tipo_de_domicilio', 'cor_ou_raca', 'prop']]


pop_censo = extrair_pop_censo_api()
pop_urbana_rural = extrair_pop_urbana_rural_ipedatapy()
pop_envelhecimento = extrair_pop_envelhecimento_api()
pop_ativa_inativa_prop = extrair_dependencia_demografica_api()
pop_genero = extrair_pop_genero_api()
frequenta_escola = extrair_6_17_frequenta_escola_api()
domicilios = extrair_moradores_domicilio_api()

renda2010 = extrair_renda_domicilio_api_2010()
renda2022 = extrair_renda_domicilio_api_2022()
renda2022['ano'] = 2022
renda2010['ano'] = 2010
renda = pd.concat([renda2010, renda2022], axis=0)

fecundidade = extrair_fecundidade_api()
domicilios_cor_raca = extrair_moradores_domicilio_cor_raca_api()

StatementMeta(, 4c740358-59e2-4822-b0f1-d841ed2233bf, 3, Submitted, Running, Running)

In [3]:
# salvar dataframes em CSV na pasta Files/gold_censo_demografico
pop_censo.to_csv(
    "/lakehouse/default/Files/gold_censo_demografico/gold_censo_piramide_populacao.csv",
    sep=";",
    index=False
)
pop_urbana_rural.to_csv(
    "/lakehouse/default/Files/gold_censo_demografico/gold_censo_populacao_urbana_rural.csv",
    sep=";",
    index=False
)
pop_envelhecimento.to_csv(
    "/lakehouse/default/Files/gold_censo_demografico/gold_censo_envelhecimento_populacional.csv",
    sep=";",
    index=False
)
pop_ativa_inativa_prop.to_csv(
    "/lakehouse/default/Files/gold_censo_demografico/gold_censo_populacao_ativa_inativa.csv",
    sep=";",
    index=False
)
pop_genero.to_csv(
    "/lakehouse/default/Files/gold_censo_demografico/gold_censo_populacao_genero.csv",
    sep=";",
    index=False
)
frequenta_escola.to_csv(
    "/lakehouse/default/Files/gold_censo_demografico/gold_censo_frequenta_escola.csv",
    sep=";",
    index=False
)
domicilios.to_csv(
    "/lakehouse/default/Files/gold_censo_demografico/gold_censo_domicilios.csv",
    sep=";",
    index=False
)
renda.to_csv(
    "/lakehouse/default/Files/gold_censo_demografico/gold_censo_renda.csv",
    sep=";",
    index=False
)
fecundidade.to_csv(
    "/lakehouse/default/Files/gold_censo_demografico/gold_censo_fecundidade.csv",
    sep=";",
    index=False
)
domicilios_cor_raca.to_csv(
    "/lakehouse/default/Files/gold_censo_demografico/gold_censo_domicilios_cor_raca.csv",
    sep=";",
    index=False
)

StatementMeta(, b45dc78c-b416-4ed9-a928-8be8482f4088, 5, Finished, Available, Finished)